In this notebook, we'll create several songs that use a typical "Barry" chord sequence, using different paramters.

# Trying out different backends

In [1]:
chord_sequence = '/Users/thorwhalen/Downloads/6dim.html'
repeats = 2
bpm = 100
target_folder = '/Users/thorwhalen/Downloads/'

In [2]:
import re
import os
from pathlib import Path
from pyRealParser import Tune
from accompy import generate_accompaniment, Score

def parse_ireal_html(html_path):
    """Extract Score from an iReal Pro HTML export file."""
    with open(html_path) as f:
        html = f.read()
    match = re.search(r'href="(irealb://[^"]+)"', html)
    if not match:
        raise ValueError("No iReal URL found in HTML file")
    url = match.group(1)
    # pyRealParser needs the raw URL; reconstruct tune string skipping empty fields
    from urllib.parse import unquote
    content = unquote(url).split('://', 1)[1]
    parts = content.split('=')
    # URL format: title=composer==style=key=n=chorddata=compstyle=bpm=repeats
    # Skip the empty part[2] and the number part[5] to match pyRealParser's expected format
    tune_str = '='.join([parts[0], parts[1], parts[3], parts[4], parts[6], parts[7], parts[8], parts[9]])
    tune = Tune(tune_str)
    measures = [[chord] for chord in tune.measures_as_strings if chord]
    return Score(
        measures=measures,
        title=tune.title or "Untitled",
        key=tune.key or "C",
        time_signature=(4, 4),
    )

score = parse_ireal_html(chord_sequence)
print(f"Title: {score.title}, Key: {score.key}")
print(f"Measures ({len(score.measures)}): {score.measures}")

Title: 6dim, Key: C
Measures (16): [['C6'], ['Do'], ['C6/E'], ['Fo'], ['C6/G'], ['G#o'], ['A-'], ['Bo'], ['C6'], ['Do'], ['C6/E'], ['Fo'], ['C6/G'], ['G#o'], ['A-'], ['Bo']]


In [ ]:
backends = ['mma', 'builtin']
results = {}

for backend in backends:
    filename = f"barry__{backend}.wav"
    output_path = os.path.join(target_folder, filename)
    try:
        audio = generate_accompaniment(
            score,
            style='swing',
            tempo=bpm,
            repeats=repeats,
            output_path=output_path,
            backend=backend,
        )
        size = os.path.getsize(audio)
        results[backend] = f"OK ({size:,} bytes)"
        print(f"{backend}: {audio} ({size:,} bytes)")
    except Exception as e:
        results[backend] = f"FAILED: {e}"
        print(f"{backend}: FAILED - {e}")

print("\nSummary:")
for b, r in results.items():
    print(f"  {b}: {r}")

# Barry sequence with different combos of tempo and styles

In [8]:
chord_sequence = '/Users/thorwhalen/Downloads/6dim.html'
repeats = 12
keys = ['Eb', 'Bb', 'F', 'C', 'G', 'D', 'A']
bpms = [80, 90, 100, 110, 120]
target_folder = '/Users/thorwhalen/Downloads/barry_variations/midi_wavs'

In [9]:
# verify resources are present
import os 

if chord_sequence.startswith(os.path.sep):
    assert os.path.isfile(chord_sequence), f"Chord sequence file not found: {chord_sequence}"

assert os.path.isdir(target_folder), f"Target folder not found: {target_folder}"

In [10]:
import re
from accompy.main import _score_to_mma, _ireal_chord_to_mma, _ensure_mma_grooves

# --- Transposition ---
_NOTES_SHARP = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
_NOTES_FLAT = ['C', 'Db', 'D', 'Eb', 'E', 'F', 'Gb', 'G', 'Ab', 'A', 'Bb', 'B']

def _note_to_index(note):
    """Convert note name (e.g. 'C', 'Eb', 'F#') to semitone index 0-11."""
    for notes in [_NOTES_SHARP, _NOTES_FLAT]:
        if note in notes:
            return notes.index(note)
    raise ValueError(f"Unknown note: {note}")

def _index_to_note(idx, use_flat=False):
    notes = _NOTES_FLAT if use_flat else _NOTES_SHARP
    return notes[idx % 12]

def _transpose_chord(chord, semitones, use_flat=False):
    """Transpose a single chord symbol by N semitones."""
    m = re.match(r'^([A-G][#b]?)(.*)', chord)
    if not m:
        return chord
    root, quality = m.group(1), m.group(2)
    # Handle slash chords
    slash_root = None
    if '/' in quality:
        parts = quality.rsplit('/', 1)
        quality = parts[0]
        slash_m = re.match(r'^([A-G][#b]?)(.*)', parts[1])
        if slash_m:
            slash_root = slash_m.group(1)
            slash_suffix = slash_m.group(2)
    new_root = _index_to_note(_note_to_index(root) + semitones, use_flat)
    result = new_root + quality
    if slash_root:
        new_slash = _index_to_note(_note_to_index(slash_root) + semitones, use_flat)
        result += '/' + new_slash + slash_suffix
    return result

def transpose_score(score, target_key):
    """Transpose a Score to a new key."""
    semitones = _note_to_index(target_key) - _note_to_index(score.key)
    use_flat = 'b' in target_key or target_key in ['F', 'Bb', 'Eb', 'Ab', 'Db']
    new_measures = [
        [_transpose_chord(c, semitones, use_flat) for c in measure]
        for measure in score.measures
    ]
    return Score(
        measures=new_measures,
        title=f"{score.title} ({target_key})",
        key=target_key,
        time_signature=score.time_signature,
    )

# Test transposition
base_score = parse_ireal_html(chord_sequence)
for k in keys:
    ts = transpose_score(base_score, k)
    print(f"Key {k}: {[m[0] for m in ts.measures]}")

Key Eb: ['Eb6', 'Fo', 'Eb6/G', 'Abo', 'Eb6/Bb', 'Bo', 'C-', 'Do', 'Eb6', 'Fo', 'Eb6/G', 'Abo', 'Eb6/Bb', 'Bo', 'C-', 'Do']
Key Bb: ['Bb6', 'Co', 'Bb6/D', 'Ebo', 'Bb6/F', 'Gbo', 'G-', 'Ao', 'Bb6', 'Co', 'Bb6/D', 'Ebo', 'Bb6/F', 'Gbo', 'G-', 'Ao']
Key F: ['F6', 'Go', 'F6/A', 'Bbo', 'F6/C', 'Dbo', 'D-', 'Eo', 'F6', 'Go', 'F6/A', 'Bbo', 'F6/C', 'Dbo', 'D-', 'Eo']
Key C: ['C6', 'Do', 'C6/E', 'Fo', 'C6/G', 'G#o', 'A-', 'Bo', 'C6', 'Do', 'C6/E', 'Fo', 'C6/G', 'G#o', 'A-', 'Bo']
Key G: ['G6', 'Ao', 'G6/B', 'Co', 'G6/D', 'D#o', 'E-', 'F#o', 'G6', 'Ao', 'G6/B', 'Co', 'G6/D', 'D#o', 'E-', 'F#o']
Key D: ['D6', 'Eo', 'D6/F#', 'Go', 'D6/A', 'A#o', 'B-', 'C#o', 'D6', 'Eo', 'D6/F#', 'Go', 'D6/A', 'A#o', 'B-', 'C#o']
Key A: ['A6', 'Bo', 'A6/C#', 'Do', 'A6/E', 'Fo', 'F#-', 'G#o', 'A6', 'Bo', 'A6/C#', 'Do', 'A6/E', 'Fo', 'F#-', 'G#o']


In [11]:
from itertools import cycle

# Jazz-appropriate MMA grooves to cycle through
jazz_grooves = [
    'Swing', 'BossaNova', 'Bebop', 'SlowJazz', 'FastSwing',
    'JazzCombo', 'JazzGuitar', 'MellowJazz', 'NiteJazz', 'ModernJazz',
    'JazzWaltz', 'EasySwing', 'GypsyJazz', 'FolkyJazzPiano',
]

# Build 14 combos: 7 keys × 2 (cycling through bpms and grooves)
combos = []
groove_cycle = cycle(jazz_grooves)
bpm_cycle = cycle(bpms)
for key in keys:
    for _ in range(2):
        combos.append((key, next(bpm_cycle), next(groove_cycle)))

print(f"Will generate {len(combos)} combos:")
for key, bpm, groove in combos:
    print(f"  Key={key}, BPM={bpm}, Groove={groove}")

Will generate 14 combos:
  Key=Eb, BPM=80, Groove=Swing
  Key=Eb, BPM=90, Groove=BossaNova
  Key=Bb, BPM=100, Groove=Bebop
  Key=Bb, BPM=110, Groove=SlowJazz
  Key=F, BPM=120, Groove=FastSwing
  Key=F, BPM=80, Groove=JazzCombo
  Key=C, BPM=90, Groove=JazzGuitar
  Key=C, BPM=100, Groove=MellowJazz
  Key=G, BPM=110, Groove=NiteJazz
  Key=G, BPM=120, Groove=ModernJazz
  Key=D, BPM=80, Groove=JazzWaltz
  Key=D, BPM=90, Groove=EasySwing
  Key=A, BPM=100, Groove=GypsyJazz
  Key=A, BPM=110, Groove=FolkyJazzPiano


In [12]:
import subprocess
import tempfile
import wave
from accompy.main import _find_mma

_ensure_mma_grooves()
mma_cmd = _find_mma()

base_score = parse_ireal_html(chord_sequence)
results = []
failed = []

for key, tempo, groove in combos:
    transposed = transpose_score(base_score, key)
    safe_key = key.replace('#', 'sharp').replace('b', 'flat')
    filename = f"barry_{safe_key}_{tempo}bpm_{groove}.wav"
    output_path = os.path.join(target_folder, filename)
    
    # Build MMA file directly (to use arbitrary groove names)
    lines = [
        f"// Generated by accompy - Barry 6dim in {key}",
        f"Tempo {tempo}",
        f"Groove {groove}",
        "",
    ]
    # Add measures (repeated)
    bar_num = 1
    for rep in range(repeats):
        for measure in transposed.measures:
            chords_mma = " ".join(_ireal_chord_to_mma(c) for c in measure)
            lines.append(f"{bar_num} {chords_mma}")
            bar_num += 1
    
    mma_content = "\n".join(lines)
    
    # Write MMA file, run MMA, render with FluidSynth
    mma_path = Path(tempfile.mktemp(suffix=".mma"))
    midi_path = mma_path.with_suffix(".mid")
    try:
        mma_path.write_text(mma_content)
        
        result = subprocess.run(
            [mma_cmd, str(mma_path), '-f', str(midi_path)],
            capture_output=True, text=True, check=True,
        )
        
        # Render MIDI to WAV using FluidSynth
        from accompy.synthesis.fluidsynth import FluidSynthBackend
        backend = FluidSynthBackend()
        backend.render_to_file(midi_path, Path(output_path))
        
        # Verify WAV file
        size = os.path.getsize(output_path)
        with wave.open(output_path, 'rb') as wf:
            duration = wf.getnframes() / wf.getframerate()
            channels = wf.getnchannels()
            sr = wf.getframerate()
        results.append((filename, size, f"{duration:.1f}s, {sr}Hz, {channels}ch"))
        print(f"  OK: {filename} ({size:,} bytes, {duration:.1f}s)")
        
    except subprocess.CalledProcessError as e:
        error_msg = (e.stderr or e.stdout or "").strip()
        failed.append((filename, groove, error_msg))
        print(f"  FAILED: {filename} - MMA error: {error_msg}")
    except Exception as e:
        failed.append((filename, groove, str(e)))
        print(f"  FAILED: {filename} - {e}")
    finally:
        mma_path.unlink(missing_ok=True)
        midi_path.unlink(missing_ok=True)

print(f"\n{'='*60}")
print(f"Generated: {len(results)} files")
if failed:
    print(f"Failed: {len(failed)} files")
    for name, groove, err in failed:
        print(f"  {name} (groove={groove}): {err}")

  OK: barry_Eflat_80bpm_Swing.wav (102,082,860 bytes, 578.7s)
  OK: barry_Eflat_90bpm_BossaNova.wav (90,747,180 bytes, 514.4s)
  OK: barry_Bflat_100bpm_Bebop.wav (81,761,580 bytes, 463.5s)
  OK: barry_Bflat_110bpm_SlowJazz.wav (74,670,124 bytes, 423.3s)
  OK: barry_F_120bpm_FastSwing.wav (68,372,780 bytes, 387.6s)
  OK: barry_F_80bpm_JazzCombo.wav (102,384,172 bytes, 580.4s)
  OK: barry_C_90bpm_JazzGuitar.wav (90,696,236 bytes, 514.2s)
  OK: barry_C_100bpm_MellowJazz.wav (81,980,972 bytes, 464.7s)
  OK: barry_G_110bpm_NiteJazz.wav (74,444,844 bytes, 422.0s)
  OK: barry_G_120bpm_ModernJazz.wav (68,526,124 bytes, 388.5s)
  OK: barry_D_80bpm_JazzWaltz.wav (76,812,844 bytes, 435.4s)
  OK: barry_D_90bpm_EasySwing.wav (90,716,716 bytes, 514.3s)
  OK: barry_A_100bpm_GypsyJazz.wav (81,744,428 bytes, 463.4s)
  OK: barry_A_110bpm_FolkyJazzPiano.wav (74,678,572 bytes, 423.3s)

Generated: 14 files


# Sunofied Barry

I mean, MMA is pretty good, but not AI gen good. 
So let's feed this to suno to get even better tracks!